In [4]:
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
analysis_name = "gsa-shiva-bg-lf"

root_path = "../.."
folder_path=root_path+"/results/"+analysis_name+"/"

results_df = pd.read_csv(folder_path+analysis_name+"_analysis.csv")

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

unique_name = results_df['name'].unique()

# Find unique difftime values to create a subplot for each
unique_difftimes = results_df['difftime'].unique()
num_plots = len(unique_difftimes)

# Create a subplot figure
fig = make_subplots(
    rows=1, cols=num_plots,
    specs=[[{'type': 'scatter3d'}] * num_plots],
    subplot_titles=[f"difftime = {d:.2e}" for d in unique_difftimes],
    horizontal_spacing=0,  # Set horizontal spacing to 0
    vertical_spacing=0     # Set vertical spacing to 0
)

min_vfinal = results_df['Vfinal'].min()
max_vfinal = results_df['Vfinal'].max()
max_vfinal = 1e0

# Iterate and add a trace for each difftime value
for i, d in enumerate(unique_difftimes, start=1):
    subset_df = results_df[results_df['difftime'] == d].copy()
    
    # Add a 3D scatter plot for the current subset
    trace = go.Scatter3d(
        x=subset_df['alpha'],
        y=subset_df['epsilon'],
        z=subset_df['beta'],
        mode='markers',
        marker=dict(
            size=4,
            color=np.log10(subset_df['Vfinal']), # Use log10 of Vfinal for color mapping
            colorscale='Viridis',
            colorbar=dict(
                title='log10 of $V_{final}/V_{0}$',
                # tickvals=np.log10(np.logspace(np.floor(np.log10(min_vfinal)), np.ceil(np.log10(max_vfinal)), 5)),
                # tickformat=".0e"
                # ticktext=[f"10^{{{int(x)}}}" for x in np.log10(np.logspace(np.floor(np.log10(min_vfinal)), np.ceil(np.log10(max_vfinal)), 5))]
            ),
            cmin=np.log10(min_vfinal),
            cmax=np.log10(max_vfinal)
        )
    )
    
    fig.add_trace(trace, row=1, col=i)
    
    # Set axes to log10
    fig.update_scenes(
        xaxis_title="Alpha",
        yaxis=dict(type='log', title='Epsilon', tickformat=".0e", nticks=4),
        zaxis=dict(type='log', title='Beta', tickformat=".0e", nticks=4),
        row=1, col=i,
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5) # Adjust these values as needed
        )
    )

fig.update_layout(
    title_text=unique_name[0],
    height=600,
    showlegend=False,
    width=2000,
)

fig.show()

In [7]:
# conda activate plotly
# cd C:\Users\stear\.conda\envs\plotly\Scripts
# python choreo_get_chrome


fig.write_image(folder_path+analysis_name+"_analysis.png",scale=1)
fig.write_image(folder_path+analysis_name+"_analysis.svg",scale=1)